# v3 · Agentic RAG in LangGraph (chapter 4)

v2 searches, but trusts whatever comes back. v3 gives retrieval judgment: **route** (search or not),
**retrieve**, **grade** the chunks, **rewrite** and retry when they are weak, and **verify** the
answer. Every loop is bounded by a counter in state (slides 48–53).

Then the demo from slide 54, four questions that exercise every branch:
"Hi!" → no retrieval · "Q3 margin?" → 1 search · "Q3 vs Q2, and why?" → 2 searches + grade ·
a vague question → rewrite + retry.

The graph built here is `build_v3_graph` in `agents/agentic_rag.py`, which chapter 5 serves on
AgentShip unchanged.

**Setup:** see `README.md` (`pip install -e .`, then `.env` names the model and server). Run with the *Agentic RAG demos* kernel.

In [ ]:
import json
import re

import httpx
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from tools.search_docs import format_hits, load_index
from tools.search_docs import search_docs as search_index
from tools.settings import API_BASE, CHAT_MODEL, HEADERS, LLM_KWARGS, RERANK_MODEL

chunks, vectors = load_index()
model = init_chat_model(CHAT_MODEL, **LLM_KWARGS)
print(f"Model: {CHAT_MODEL}   Grader: {RERANK_MODEL or CHAT_MODEL}   Index: {len(chunks)} chunks")

## RAG is just another tool (slides 48 and 51)

The description **is** the routing policy: what the documents hold, and when *not* to search.

In [ ]:
@tool
def search_docs(query: str, k: int = 5) -> str:
    """Search Acme's internal filings (the Q3 2026 10-Q, with Q2 2026 results in an appendix).
    Use for Acme-specific numbers. One quarter or topic per search: to compare two quarters,
    call it once for each. Do NOT use for definitions, small talk or live prices."""
    return format_hits(search_index(query, k))


@tool
def web_search(query: str) -> str:
    """Search the live web, for news and prices that change daily."""
    return "web_search is not connected in this lab."

### Call RAG or not? Let the LLM decide (slide 51)

Bind both tools and look only at the model's decision. Nothing is searched yet.

In [ ]:
router = model.bind_tools([search_docs, web_search])
for q in ["Thanks, that helps!", "What is a P/E ratio?", "What was our Q3 revenue?",
          "Compare Q3 vs Q2 margins", "Today's Acme stock price?"]:
    calls = router.invoke(q).tool_calls
    decision = ", ".join(f"{c['name']}({c['args'].get('query', '')!r})" for c in calls) or "answer directly"
    print(f"{q:<28} → {decision}")

## The graph: route, retrieve, grade, retry, verify (slides 52–53)

The state is `MessagesState` plus counters. `retries` is what makes the loop end.

In [ ]:
PROMPT = """You answer questions about Acme Corp's quarterly filing (Form 10-Q for Q3 2026).
Use search_docs for any Acme-specific fact or number, once per quarter or topic you need: to
compare two quarters, search for each. Answer greetings, thanks and general definitions directly,
without searching. Answer only from the search results, cite chunk ids like [chunk 36], and say
so when the filings don't contain the answer. Be concise."""

MAX_RETRIES = 2  # rewrites per question
MAX_SEARCHES = 4  # search rounds per question, a second cap so the loop always ends
SEPARATOR = "\n\n---\n\n"  # between chunks in a search_docs result


class RAGState(MessagesState):
    retries: int  # rewrites so far this question: the counter that ends the loop
    relevant: int  # chunks the grader kept from the last search
    note: str | None  # a one-off instruction for the agent's next call, from verify
    query: str | None  # the rewritten query the agent must search with next
    verified: bool

In [ ]:
def _text(message) -> str:
    return message.content if isinstance(message.content, str) else str(message.content)


def _this_turn(messages: list) -> list:
    """Everything from the latest user message on."""
    start = max(i for i, m in enumerate(messages) if isinstance(m, HumanMessage))
    return messages[start:]

In [ ]:
tools = [search_docs]
llm = model.bind_tools(tools)
judge = model.bind(temperature=0)  # the small decisions should not be a coin toss


def ask(prompt: str) -> str:  # the small decisions: grade, rewrite, verify
    return _text(judge.invoke([HumanMessage(content=prompt)])).strip()

**The grader** · a small model scores each chunk against the query that found it. A cross-encoder reranker reads the pair together; above 0 means relevant.

In [ ]:
def relevant(query: str, passages: list[str], ask) -> list[bool]:
    """The grader: which passages state what the search looked for?

    A cross-encoder (DEMO_RERANK_MODEL) reads each query–passage pair and scores it: above 0 means
    relevant. It is small, fast and steadier than asking the chat model. Without one, the chat model
    picks the passages instead.
    """
    if RERANK_MODEL:
        resp = httpx.post(f"{API_BASE}/reranking", headers=HEADERS, timeout=60,
                          json={"model": RERANK_MODEL, "query": query, "documents": passages})
        resp.raise_for_status()
        scores = {r["index"]: r["relevance_score"] for r in resp.json()["results"]}
        return [scores[i] > 0 for i in range(len(passages))]
    listing = "\n\n".join(f"<passage {n}>\n{p}\n</passage {n}>" for n, p in enumerate(passages, 1))
    verdict = ask(f"Search: {query}\n\n{listing}\n\nWhich passages state what the search looks for? "
                  f'Reply with JSON only, like {{"relevant": [1, 3]}}, or {{"relevant": []}}.')
    try:
        keep = {int(n) for n in json.loads(re.search(r"\{.*\}", verdict, re.S).group())["relevant"]}
    except Exception:  # an unreadable verdict: keep everything rather than lose evidence
        keep = set(range(1, len(passages) + 1))
    return [n in keep for n in range(1, len(passages) + 1)]

**agent** · the LLM with `search_docs` bound. A new question resets the counters. After `MAX_SEARCHES` rounds it gets no tools, so the loop has to end.

In [ ]:
def agent(state: RAGState) -> dict:
    """LLM + search_docs: decides whether to search, what for, and when to answer."""
    messages = state["messages"]
    new_turn = isinstance(messages[-1], HumanMessage)
    note = None if new_turn else state.get("note")
    searches = sum(1 for m in _this_turn(messages) if isinstance(m, AIMessage) and m.tool_calls)
    system = PROMPT + (f"\n\n{note}" if note else "")
    history = [m for m in messages if not isinstance(m, SystemMessage)]
    if state.get("query") and not new_turn and searches < MAX_SEARCHES:  # after a rewrite
        call = {"name": "search_docs", "args": {"query": state["query"]}, "id": f"rewrite-{state['retries']}"}
        reply = AIMessage(content="", tool_calls=[call])
    elif searches < MAX_SEARCHES:
        reply = llm.invoke([SystemMessage(content=system)] + history)
    else:  # out of searches: no tools bound, so say why, or the model writes a search as text
        system += "\n\nYou have used every search for this question. Answer from the results above."
        reply = model.invoke([SystemMessage(content=system)] + history)
        if "<tool_call>" in _text(reply):
            answer = _text(reply).split("<tool_call>")[0].strip()
            reply = AIMessage(content=answer or "The filings' search results don't answer this.")
    update = {"messages": [reply], "note": None, "query": None}
    if new_turn:
        update |= {"retries": 0, "relevant": 0, "verified": False}
    return update

**grade** · one call to the model: which chunks help? The rest are dropped from the context.

In [ ]:
def grade(state: RAGState) -> dict:
    """Small model: which chunks state what each search looked for? Drop the rest."""
    messages = state["messages"]
    results = []
    for m in reversed(messages):  # the ToolMessages this search just added
        if not isinstance(m, ToolMessage):
            break
        results.insert(0, m)
    call = next(m for m in reversed(messages) if isinstance(m, AIMessage) and m.tool_calls)
    queries = {c["id"]: c["args"].get("query", "") for c in call.tool_calls}
    graded, n_relevant = [], 0
    for m in results:  # each search is graded against its own query
        passages = [p for p in _text(m).split(SEPARATOR) if p.strip()]
        kept = [p for p, ok in zip(passages, relevant(queries.get(m.tool_call_id, ""), passages, ask)) if ok]
        n_relevant += len(kept)
        graded.append(ToolMessage(content=SEPARATOR.join(kept) or "No relevant passages.",
                                  tool_call_id=m.tool_call_id, name=m.name, id=m.id))
    return {"messages": graded, "relevant": n_relevant}

**rewrite** · nothing relevant: write a better query, and add one to `retries`.

In [ ]:
def rewrite(state: RAGState) -> dict:
    """Nothing relevant: write a better query, count one retry."""
    messages = state["messages"]
    question = _text(_this_turn(messages)[0])
    last = next(m for m in reversed(messages) if isinstance(m, AIMessage) and m.tool_calls)
    queries = [c["args"].get("query", "") for c in last.tool_calls]
    better = ask(f"Question: {question}\nThese searches of a company's 10-Q filing found nothing "
                 f"relevant: {queries}\nWrite one better search query, in the words a 10-Q would "
                 f"use. Reply with the query only.").splitlines()[0][:200]
    return {"retries": state.get("retries", 0) + 1, "query": better}

**give_up** · out of retries: say so instead of guessing.

In [ ]:
def give_up(state: RAGState) -> dict:
    """Out of retries: say so instead of guessing."""
    return {"messages": [AIMessage(content="I couldn't find that in Acme's filings, even after "
                                           "rewriting the search.")]}

**verify** · is every claim in the answer in the evidence? If not, answer once more.

In [ ]:
def check(state: RAGState) -> dict:
    """Verify: is every claim in the answer in the evidence? If not, answer once more."""
    messages = state["messages"]
    evidence = "\n\n".join(_text(m) for m in _this_turn(messages) if isinstance(m, ToolMessage))
    if state.get("verified") or not evidence:  # already checked, or nothing was searched
        return {}
    answer = messages[-1]
    verdict = ask(f"EVIDENCE:\n{evidence}\n\nANSWER:\n{_text(answer)}\n\nIs every number and factual "
                  f"claim in ANSWER supported by EVIDENCE? Saying the evidence doesn't contain something counts as "
                  f"supported. Reply with yes or no only.")
    if verdict.strip("*_ .").lower().startswith("yes"):
        return {"verified": True}
    return {"verified": True, "messages": [RemoveMessage(id=answer.id)],
            "note": "Your last answer had claims that are not in the search results. Answer again "
                    "using only the search results, or say the filings don't contain it."}

**The decision after grading** (slide 53).

In [ ]:
def after_grade(state: RAGState) -> str:
    if state["relevant"]:
        return "agent"  # good: answer
    if state.get("retries", 0) < MAX_RETRIES:
        return "rewrite"  # weak: try again
    return "give_up"  # out of retries

### Wire it up

Same building blocks as chapter 3: nodes, a router, and edges back. `tools_condition` routes
after the agent, exactly as in v2; `after_grade` answers, retries, or gives up.

In [ ]:
verify = True

g = StateGraph(RAGState)
g.add_node("agent", agent)  # LLM + search_docs
g.add_node("retrieve", ToolNode(tools))
g.add_node("grade", grade)  # small model: relevant?
g.add_node("rewrite", rewrite)  # better query, retries+1
g.add_node("give_up", give_up)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition, {"tools": "retrieve", END: "verify" if verify else END})
g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", after_grade, ["agent", "rewrite", "give_up"])
g.add_edge("rewrite", "agent")
g.add_edge("give_up", END)
if verify:  # slide 52's last step, before the final answer
    g.add_node("verify", check)
    g.add_conditional_edges("verify", lambda s: "agent" if s.get("note") else END, ["agent", END])

app = g.compile(checkpointer=InMemorySaver())
print(app.get_graph().draw_mermaid())

## Demo (slide 54)

Stream each run with `stream_mode="updates"` to see which path it took.

In [ ]:
from uuid import uuid4


def run(question: str, thread: str | None = None) -> None:
    """One question, on a new thread unless you pass one, so each demo starts clean."""
    print(f"› {question}\n")
    cfg = {"configurable": {"thread_id": thread or str(uuid4())}, "recursion_limit": 25}
    for step in app.stream({"messages": [("user", question)]}, cfg, stream_mode="updates"):
        for node, update in step.items():
            update = update or {}
            for m in update.get("messages", []):
                if isinstance(m, RemoveMessage):
                    print(f"  {node:>8} → unsupported claims: answer again")
                elif getattr(m, "tool_calls", None):
                    print(f"  {node:>8} → search_docs{[c['args']['query'] for c in m.tool_calls]}")
                elif isinstance(m, ToolMessage) and node == "retrieve":
                    print(f"  {node:>8} → {m.content.count('[chunk ')} chunks")
                elif not isinstance(m, ToolMessage):
                    print(f"  {node:>8} → {m.content}")
            if node == "grade":
                print(f"  {node:>8} → {update['relevant']} relevant chunks kept")
            if node == "rewrite":
                print(f"  {node:>8} → retry {update['retries']}: {update['query']!r}")
            if node == "verify" and update.get("verified") and not update.get("messages"):
                print(f"  {node:>8} → every claim is in the evidence")
    print()

In [ ]:
run("Hi!")

In [ ]:
run("What was Acme's Q3 margin?")

In [ ]:
run("How did Acme's Q3 margin compare to Q2, and why?")

A vague question, in words the filing doesn't use. The grader rejects the first search, so `rewrite` writes a better query and the agent searches again:

In [ ]:
run("How much work has Acme been contracted for but not yet delivered?")

And a question the filings can't answer: it runs out of retries and says so.

In [ ]:
run("How many people work in Acme's Tokyo office?")

## Checkpoint · v3 gets it right, in a notebook (slide 55)

| ✓ What v3 can do | ✗ What still breaks |
|---|---|
| Routes: no search for small talk | No API for real users → Ch 5 |
| Grades chunks, rewrites and retries, capped | A restart loses the conversation → Ch 5 |
| Answers only from the evidence | Can't see cost per step → Ch 5 |
| | No human approval for risky actions → Ch 5 |

Chapter 5 serves this same graph with AgentShip:

```bash
agentship run agents/agentic_rag.yaml --input "How did Acme's Q3 margin compare to Q2, and why?"
agentship serve --agents-dir agents
```